# MobileNetV2 — Training

Train a **MobileNetV2** classifier on the cleaned food dataset (80% train, 10% val, 10% test).

1. **Setup** — Paths, device, labels.
2. **Dataset & DataLoaders** — Load from `combined_index.csv` by split.
3. **Model** — Pretrained MobileNetV2 + new classifier head.
4. **Training** — Train/validate, save best checkpoint.
5. **Evaluation** — Optional test-set accuracy.

In [1]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


---
## 1. Setup

In [2]:
import json
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

BASE = Path(".").resolve()
DATA_DIR = BASE / "combined_dataset"
INDEX_CSV = DATA_DIR / "combined_index.csv"
LABELS_JSON = DATA_DIR / "labels.json"
LABEL2ID_JSON = DATA_DIR / "label2id.json"
CHECKPOINT_DIR = BASE / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

assert INDEX_CSV.exists(), f"Run DataCleaning_Exploration.ipynb first. Missing: {INDEX_CSV}"

with open(LABEL2ID_JSON, "r", encoding="utf-8") as f:
    label2id = json.load(f)
with open(LABELS_JSON, "r", encoding="utf-8") as f:
    id2label = json.load(f)

num_classes = len(label2id)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Classes: {num_classes}, Device: {device}")

Classes: 65, Device: cpu


---
## 2. Dataset & DataLoaders

In [3]:
import pandas as pd

class FoodDataset(Dataset):
    def __init__(self, index_csv, split_name, base_path, transform=None):
        self.base = Path(base_path)
        self.df = pd.read_csv(index_csv)
        self.df = self.df[self.df["split"] == split_name].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        path = Path(row["path"])
        if not path.is_absolute():
            path = self.base / path
        img = Image.open(path).convert("RGB")
        label_id = int(row["label_id"])
        if self.transform:
            img = self.transform(img)
        return img, label_id

# ImageNet-style normalization (MobileNetV2 is pretrained on ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = FoodDataset(INDEX_CSV, "train", BASE, transform=train_transform)
val_ds = FoodDataset(INDEX_CSV, "val", BASE, transform=eval_transform)
test_ds = FoodDataset(INDEX_CSV, "test", BASE, transform=eval_transform)

batch_size = 32
num_workers = 0  # set to 2 or 4 if you have no Windows multiprocessing issues

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Train: 43340, Val: 5417, Test: 5418


---
## 3. Model

In [4]:
def build_mobilenetv2(num_classes, pretrained=True):
    try:
        model = models.mobilenet_v2(weights=(models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None))
    except Exception:
        model = models.mobilenet_v2(pretrained=pretrained)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_mobilenetv2(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

print(model.classifier)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\MSI GF63/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 22.3MB/s]


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

---
## 4. Training loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in tqdm(loader, desc="Train"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

In [ ]:
epochs = 15
best_val_acc = 0.0

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
    scheduler.step(val_acc)
    print(f"  Train loss: {train_loss:.4f} acc: {train_acc:.4f}")
    print(f"  Val   loss: {val_loss:.4f} acc: {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        ckpt = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "num_classes": num_classes,
        }
        torch.save(ckpt, CHECKPOINT_DIR / "mobilenetv2_best.pt")
        print(f"  Saved best checkpoint (val_acc={val_acc:.4f})")

print(f"\nBest val accuracy: {best_val_acc:.4f}")

---
## 5. Test set evaluation (optional)

In [ ]:
# Load best checkpoint and evaluate on test set
ckpt_path = CHECKPOINT_DIR / "mobilenetv2_best.pt"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    test_loss, test_acc = eval_epoch(model, test_loader, criterion, device)
    print(f"Test loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}")
else:
    print("No checkpoint found. Run training first.")